In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.00),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  2750.00),
    ("Eve",   "Savings",   150.00),
]

columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

df.show()            # print the table
# df.printSchema()   # show column names + types
# df.count()         # count rows (returns a number)

# --- Summing a column ---
df.agg(F.sum("balance")).show()                    # total of all balances

# --- Sum with a WHERE clause ---
(df.filter(F.col("balance") > 1000.00)
   .agg(F.sum("balance"))
   .show())                                        # total of balances over 1000

# --- Sum with two conditions (AND) ---
(df.filter((F.col("balance") > 1000.00) & (F.col("account_type") == "Savings"))
   .agg(F.sum("balance"))
   .show())                                        # Savings balances over 1000

# --- Filter rows where the name contains a lowercase "e" ---
(df.filter((F.col("balance") > 1000.00) & (F.col("name").contains("e")))
   .agg(F.sum("balance"))
   .show())                                        # over-1000 with an "e" in the name

# --- Just show matching rows instead of summing ---
(df.filter(F.col("name").contains("e"))
   .show())                                        # people with an "e" in their name


In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.96),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  2750.00),
    ("Eve",   "Savings",   -150.00),
]


columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

df.select("name", "balance").show() 


(df.withColumn("with_interest", F.col("balance") * 1.05)
   .show())


(df.withColumn("tier",
        F.when(F.col("balance") >= 5000, "Gold")
         .when(F.col("balance") >= 1000, "Silver")
         .otherwise("Bronze"))
   .show())


df.select("name", "balance").show() 

df.withColumn("balance_rounded", F.round( F.col("balance"),0)).show()

df.withColumn("active", F.when (F.col("balance") > 0, "Yes")
              .otherwise("No")).show()


In [0]:
from pyspark.sql import functions as F

# --- Drill 1: create a DataFrame ---
data = [
    ("Alice", "Savings",  1200.96),
    ("Bob",   "Checking",  350.00),
    ("Carol", "Savings",  8000.00),
    ("Dave",  "Current",  12750.00),
    ("Eve",   "Savings",   -150.00),
]


columns = ["name", "account_type", "balance"]

df = spark.createDataFrame(data, columns)

(df.groupBy("account_type")
   .count()
   .show())


(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .show())


(df.groupBy("account_type")
   .agg(
       F.count("*").alias("num_accounts"),
       F.sum("balance").alias("total_balance"),
       F.round(F.avg("balance"),2).alias("avg_balance"),
       F.min("balance").alias("min_balance"),
       F.max("balance").alias("max_balance"),
   )
   .show())


(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .filter(F.col("total_balance") > 2000)
   .show())


(df.groupBy("account_type")
   .count()
   .show())



(df.groupBy("account_type")
   .agg(
       F.round (F.avg("balance"),2).alias("Avg_balance"),
       F.round(F.sum("balance"),2).alias("total_balance"),
       F.max("balance").alias("max_balance")
   )
   .show())

(df.groupBy("account_type")
   .agg(F.sum("balance").alias("total_balance"))
   .filter(F.col("total_balance") > 10000)
   .show())

In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with categories (like our 'tier' column earlier)
data = [
    ("Alice", "Gold", 6000),
    ("Bob", "Silver", 2500),
    ("Charlie", "Bronze", 500),
    ("Diana", "Gold", 8000),
    ("Evan", "Silver", 1500),
    ("Fiona", "Bronze", 800)
]

columns = ["name", "tier", "balance"]
df = spark.createDataFrame(data, columns)

# 2. Group by tier and calculate metrics (Total balance, Average balance, and Customer count)
summary_df = df.groupBy("tier").agg(
    F.sum("balance").alias("total_balance"),
    F.round(F.avg("balance"), 2).alias("avg_balance"),
    F.count("name").alias("customer_count")
)

# 3. Show the resulting aggregated DataFrame
summary_df.show()


from pyspark.sql import functions as F

# 1. Group by tier, filter for groups with more than 1 customer, and sort by total balance descending
filtered_summary_df = (
    df.groupBy("tier")
    .agg(
        F.sum("balance").alias("total_balance"),
        F.round(F.avg("balance"), 2).alias("avg_balance"),
        F.count("name").alias("customer_count")
    )
    .filter(F.col("avg_balance") >= 1000)
    .orderBy(F.col("total_balance").asc())
)

filtered_summary_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create a customer accounts DataFrame
accounts_data = [
    (101, "Alice", "Gold"),
    (102, "Bob", "Silver"),
    (103, "Charlie", "Bronze"),
    (104, "Diana", "Gold")
]
accounts_df = spark.createDataFrame(accounts_data, ["account_id", "name", "tier"])

# 2. Create a separate transactions DataFrame
transactions_data = [
    (101, 500),
    (101, 1200),
    (102, 300),
    (105, 999) # ID 105 doesn't exist in accounts
]
transactions_df = spark.createDataFrame(transactions_data, ["account_id", "transaction_amount"])

# 3. Perform a Left Join to keep all accounts and match their transactions
joined_df = accounts_df.join(
    transactions_df, 
    on="account_id", 
    how="left"
)

joined_df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create sample transaction data
data = [
    (101, "2026-09-01", 100),
    (101, "2026-09-05", 250),
    (101, "2026-09-10", 75),
    (102, "2026-09-02", 500),
    (102, "2026-09-08", 150)
]
df = spark.createDataFrame(data, ["account_id", "date", "amount"])

# 2. Define a window specification: partition by account, order by date
window_spec = Window.partitionBy("account_id").orderBy("date")

# 3. Apply window functions (Row Number and Running Total / Cumulative Sum)
ranked_df = df.withColumn("transaction_seq", F.row_number().over(window_spec)) \
              .withColumn("running_total", F.sum("amount").over(window_spec))

ranked_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with missing values (None represents NULL in Python)
data = [
    (101, "Alice", 5000, "Active"),
    (102, "Bob", None, "Pending"),
    (103, None, 1200, None),
    (104, "Diana", 3400, "Active")
]
df = spark.createDataFrame(data, ["account_id", "name", "balance", "status"])

# 2. Fill missing balances with 0 and missing names/status with defaults
cleaned_df = df.na.fill({"balance": 0.0, "name": "Unknown Customer", "status": "Inactive"})

# 3. Use Coalesce to create a fallback priority column
fallback_df = cleaned_df.withColumn(
    "display_status", 
    F.coalesce(F.col("status"), F.lit("Default Status"))
)

fallback_df.show()

In [0]:
from pyspark.sql import functions as F

data = [
    (101, "Alice", None),
    (102, "Bob", "Pending")
]
df = spark.createDataFrame(data, ["account_id", "name", "status"])

# Coalesce checks 'status' first; if it's null, it falls back to the literal string
result_df = df.withColumn(
    "display_status", 
    F.coalesce(F.col("status"), F.lit("Default Status"))
)

result_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create a small DataFrame to save
data = [(101, "Alice", 5000), (102, "Bob", 2500)]
df = spark.createDataFrame(data, ["account_id", "name", "balance"])

# 2. Write the DataFrame to a managed Delta table or file path
# (Using a temporary managed table name for easy testing in Databricks)
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("customer_balances_summary")
)

# 3. Read it right back to verify it was saved successfully
saved_df = spark.table("customer_balances_summary")
saved_df.show()

In [0]:
%sql
SELECT * FROM customer_balances_summary WHERE balance > 2000;

In [0]:
from pyspark.sql import functions as F

# 1. Create sample account data
data = [
    (101, "Active", 5000.0),
    (102, "Pending", 1500.0),
    (103, "Active", 3200.0),
    (104, "Inactive", 400.0),
    (105, "Active", 7500.0)
]
df = spark.createDataFrame(data, ["account_id", "status", "balance"])

# 2. Group by status and calculate total balance, average balance, and account count
agg_df = df.groupBy("status").agg(
    F.sum("balance").alias("total_balance"),
    F.round(F.avg("balance"), 2).alias("avg_balance"),
    F.count("account_id").alias("account_count")
)




agg_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Create customer DataFrame
customers_data = [
    (101, "Alice", "London","UK"),
    (102, "Bob", "Birmingham","France"),
    (103, "Charlie", "Manchester","Germany"),
    (104, "Diana", "Paris","Sweden")
]
customers_df = spark.createDataFrame(customers_data, ["account_id", "name", "city", "country"])

# 2. Create transaction summary DataFrame
transactions_data = [
    (101, 5000.0,"UK"),
    (102, 1500.0,"France"),
    (104, 3200.0,"Sweden")
]
transactions_df = spark.createDataFrame(transactions_data, ["account_id",  "balance", "country"])

joined_df = customers_df.join(
    transactions_df , 
    on=["account_id", "country"], 
    how="inner"  #left, right, full
)
 

joined_df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create sample transaction data
data = [
    (1, "London", 500.0),
    (2, "London", 1200.0),
    (3, "London", 800.0),
    (4, "Birmingham", 300.0),
    (5, "Birmingham", 1500.0)
]
df = spark.createDataFrame(data, ["txn_id", "branch", "amount"])

# 2. Define the window specification: partition by branch, order by amount descending - this does not hold data, 
#.   but is blueprint or the partitioniing used in the following winfoes function.
window_spec = Window.partitionBy("branch").orderBy(F.col("amount").desc())

# 3. Apply the row_number function
ranked_df = df.withColumn("rank_in_branch", F.row_number().over(window_spec))

ranked_df.show()


In [0]:
from pyspark.sql import functions as F

# 1. Create sample data with missing values (None / null)
data = [
    (101, "Alice", 5000.0),
    (102, None, 1500.0),
    (103, "Charlie", None),
    (104, "Diana", 0.0)
]
df = spark.createDataFrame(data, ["account_id", "name", "balance"])

# 2. Fill missing names with 'Unknown' and missing balances with 0.0
cleaned_df = df.fillna({
    "name": "Unknown",
    "balance": 0.0
})

cleaned_df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# 1. Create sample account data
data = [
    (101, 1500.0),
    (102, 12500.0),
    (103, 400.0),
    (104, None)
]
df = spark.createDataFrame(data, ["account_id", "balance"])

# 2. Define a standard Python function for custom business logic
def categorize_balance(balance):
    if balance is None:
        return "Unknown"
    elif balance > 10000.0:
        return "High-Net-Worth"
    elif balance > 1000.0:
        return "Standard"
    else:
        return "Basic"

# 3. Register the function as a PySpark UDF, specifying the return type
balance_category_udf = F.udf(categorize_balance, StringType())

# 4. Apply the UDF to the DataFrame
result_df = df.withColumn("category", balance_category_udf(F.col("balance")))

result_df.show()

In [0]:
from pyspark.sql import functions as F

# 1. Generate sample data
data = [(i, f"Category_{i % 3}", float(i * 10)) for i in range(1000)]
df = spark.createDataFrame(data, ["id", "category", "amount"])

# 2. Check the current partition count
print(f"Initial partition count: {df.rdd.getNumPartitions()}")

# 3. Repartition into 4 partitions to parallelize downstream work
repartitioned_df = df.repartition(4, "category")
print(f"New partition count: {repartitioned_df.rdd.getNumPartitions()}")

# 4. Coalesce down to 1 partition cleanly before writing to storage
final_df = repartitioned_df.coalesce(1)
print(f"Final partition count: {final_df.rdd.getNumPartitions()}")

# PySpark Partitioning: Core Concepts & Architecture

Partitioning in PySpark operates on two entirely distinct levels: **in-memory parallelisation** and **storage-level organisation**. Understanding the separation between logical execution chunks and physical file structures is essential for building scalable cloud data pipelines.

---

## 1. In-Memory Partitions (DataFrame Partitions)

* **Logical Chunks, Not Physical Nodes:** Partitions are logical divisions of a DataFrame distributed across worker nodes by Spark's scheduler. Having a cluster with a specific number of nodes does not dictate your partition count; partitions represent the atomic units of work assigned to CPU cores.
* **Analogy to SQL Server:** In-memory partitioning behaves similarly to a parallel query execution plan (comparable to `MAXDOP > 1` in SQL Server), splitting work across multiple worker threads to process datasets concurrently.
* **Controlling Partitions:**
* **`.repartition(n)`:** Triggers a full network shuffle to redistribute data evenly across a specified number of partitions or group them by a specific column key.
* **`.coalesce(n)`:** Reduces the number of partitions locally *without* a full network shuffle, making it ideal for clearing small file clutter before writing to storage.



---

## 2. Storage-Level Partitioning (`.partitionBy`)

* **Physical Layout:** When writing a DataFrame out to storage (such as Delta Lake or Parquet) using `.write.partitionBy("column_name")`, Spark physically organises the files into a directory hierarchy on disk (e.g., `/country=UK/`, `/country=US/`).
* **Analogy to SQL Server:** This is the direct distributed equivalent of physical table partitioning or partition elimination in SQL Server.
* **Partition Pruning:** When downstream queries include a filter on the partitioned column (e.g., `.filter(F.col("country") == "UK")`), Spark's optimiser automatically skips scanning irrelevant folders and reads *only* the specific subfolder containing matching records.

---

## 3. Serverless Compute Limitations

* **Restriction on Low-Level APIs:** When operating on modern serverless compute environments (such as Databricks Serverless), low-level RDD methods (e.g., `.rdd` or `sc.parallelize()`) are restricted due to multi-tenant isolation and managed infrastructure abstractions.
* **DataFrame-Native Alternatives:** For custom, row-level transformations at the partition level without performance bottlenecks, modern distributed workflows rely on DataFrame-native functions like **`mapInPandas()`**, which processes iterators of Pandas DataFrames per partition safely within the supported API.


# PySpark Performance Tuning & Shuffling: Understanding the Engine

When transitioning from SQL Server to PySpark, the biggest mindset shift isn't just learning a new syntax—it is learning how to think about **distributed compute**.

In SQL Server, the query optimizer handles execution plans, indexing strategies, and parallel threads under the hood. In PySpark, while the Catalyst Optimizer does heavy lifting, **how you write your transformations directly dictates whether your cluster runs efficiently or grinds to a halt due to network bottlenecks.**

---

## 1. Narrow vs. Wide Transformations (The Core Bottleneck)

Understanding performance in Spark starts with how operations are categorized:

* **Narrow Transformations (Fast & Scalable):**
* *What it is:* Data required to compute the output lives in a single partition. No data moves across the network.
* *Examples:* `.filter()`, `.select()`, `.withColumn()`, `.drop()`.
* *SQL Server Equivalent:* Local row-by-row operations or index seeks where no cross-thread coordination is required.


* **Wide Transformations (Heavy & Expensive):**
* *What it is:* Data from *multiple* partitions must be brought together and redistributed across the cluster. This triggers a **Shuffle**.
* *Examples:* `.groupBy()`, `.join()`, `.distinct()`, `.repartition()`.
* *SQL Server Equivalent:* A massive `HASH MATCH JOIN` or `STREAM AGGREGATE` that requires spilling data to `tempdb` across nodes.



---

## 2. What is a "Shuffle" and Why Does it Hurt?

When Spark executes a wide transformation (like a join or aggregation), it must perform a **Shuffle**.

1. **Exchange Phase:** Worker nodes write intermediate data to local disk.
2. **Network Phase:** Other worker nodes pull chunks of data across the network so that all rows with the same grouping/join key end up on the exact same executor.

**Why it hurts performance:** Network I/O and disk I/O are the slowest bottlenecks in distributed computing. Minimizing shuffles is the #1 rule of PySpark optimization.

---

## 3. Key Optimization Strategies

### A. Broadcast Joins (Eliminating Shuffles for Small Tables)

If you are joining a massive fact table (millions of rows) with a small dimension table (e.g., a list of 50 branches or currency codes), a standard join forces a full network shuffle of *both* tables.

Instead, you can **broadcast** the small table. Spark sends a copy of the small table to every single worker node in advance, allowing each node to perform a local join instantly with zero network shuffle.

```python
from pyspark.sql import functions as F

# Force a broadcast join on the smaller DataFrame
optimized_df = large_fact_df.join(
    F.broadcast(small_dim_df), 
    "branch_id"
)

```

*(SQL Server Equivalent: Similar in concept to a `LOOP JOIN` with a small table hint, caching pages in memory).*

### B. Avoiding Data Skew

If your partition key is unevenly distributed (e.g., 90% of your transactions belong to a single high-volume retailer or region, and 10% are split across 100 others), a shuffle will cause **stragglers**. 99 of your executors will finish in seconds, while 1 executor sits processing 90% of the data.

* **Solution:** Salting keys or pre-filtering out heavily skewed null/default values before performing aggregations or joins.

### C. Coalesce vs. Repartition (Managing Output Files)

* Use **`.repartition(n)`** when you *want* a full shuffle to balance out skewed data or increase parallelism before a heavy operation.
* Use **`.coalesce(n)`** when you want to *reduce* the number of partitions (e.g., before writing to storage) **without** a shuffle. It merges existing partitions locally, preventing the creation of millions of tiny, inefficient files on your data lake.


# Delta Lake Architecture: Core Components & Mechanics

Delta Lake is an open-source storage layer that brings ACID (Atomicity, Consistency, Isolation, Durability) transactions, reliability, and performance to data lakes. Built directly on top of standard cloud object storage and open file formats like Apache Parquet, it serves as the foundational storage format for modern data platforms like Databricks and Microsoft Fabric.

---

## 1. The Core Components of Delta Lake

A Delta Lake table is not a proprietary database file; rather, it consists of three primary pillars stored in your cloud storage bucket (Azure Blob Storage, ADLS Gen2, AWS S3, or Google Cloud Storage):

* **Parquet Data Files:** The actual raw data is stored in standard, highly compressed columnar Apache Parquet files. When you insert, update, or append data, new Parquet files are written to the directory.
* **The Transaction Log (`_delta_log`):** Housed inside a hidden `.delta_log` folder alongside your data files, this is the brain of Delta Lake. It records an ordered, immutable JSON log of every single transaction that has ever occurred on the table.
* **Metadata & Statistics:** The transaction log records file additions, removals, min/max statistics for partition pruning, and data skipping metadata.

---

## 2. How ACID Transactions Work (The Transaction Log)

In a traditional data lake (raw Parquet or CSV), if a write job fails halfway through, you are left with corrupted data files, partial writes, or phantom reads. Delta Lake solves this using **Optimistic Concurrency Control (OCC)** managed through the transaction log:

1. **Commit Log Files (`000000.json`, `000001.json`, etc.):** Every atomic write (insert, update, delete) creates a sequentially numbered JSON file in the `_delta_log` directory.
2. **Atomic Commits:** An operation is only considered successful when its corresponding JSON commit file is written to storage.
3. **State Reconstruction:** When a query engine reads a Delta table, it reads the transaction log from the beginning (or a checkpoint) to determine *which* Parquet files are active and valid at that exact moment. If two writers attempt to modify the table simultaneously, Delta uses OCC to check for conflicts; if none exist, both succeed sequentially, otherwise one retries.

---

## 3. Key Advanced Capabilities Enabled by the Architecture

### A. Time Travel (Data Versioning)

Because the transaction log tracks every change as an immutable ledger, Delta Lake keeps a history of your data. You can query or rollback to previous versions of a table using a timestamp or version number:

```python
# Query data as it looked at a specific version
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/path/to/table")

# Query data as of a specific timestamp
df_time = spark.read.format("delta").option("timestampAsOf", "2026-06-01T00:00:00Z").load("/path/to/table")

```

### B. Scalable Metadata & Data Skipping

Unlike traditional data lakes that must list every physical file in a directory to understand the schema and data bounds, Delta Lake reads the transaction log metadata. Furthermore, because it automatically collects statistics (minimum and maximum values) for columns during writes, the query engine performs **data skipping**, completely ignoring Parquet files that do not contain data matching your `WHERE` clauses.

### C. Compaction and Optimization (`OPTIMIZE`)

Frequent small writes (such as streaming data or micro-batching) can create the dreaded **"small file problem"**, which severely degrades query performance due to excessive file metadata overhead. Delta Lake allows you to run an compaction command:

```sql
OPTIMIZE table_name;

```

This background job reads small Parquet files and rewrites them into larger, optimal-sized files (typically around 1 GB) without changing the logical table state or disrupting readers.

### D. Data Maintenance (`VACUUM`)

Over time, updates, deletes, and optimizations leave behind older versions of Parquet files that are no longer referenced by the active transaction log. To reclaim storage space and purge old historical versions beyond your retention window, you use the vacuum command:

```sql
-- Removes files older than 7 days (default retention)
VACUUM table_name RETAIN 168 HOURS;

```

*(Note: Running a strict vacuum removes the ability to time-travel back prior to that retention threshold).*

---

How does the transactional logging model of Delta Lake compare to how you handle transaction logging and recovery in SQL Server environments?

# Azure Data Engineering & Orchestration

In modern cloud data architectures, building robust transformations using PySpark or Delta Lake is only half the battle. You also need an orchestration layer to automate, schedule, monitor, and chain these workloads together into reliable data pipelines.

In the Microsoft ecosystem, this orchestration layer is handled primarily by **Azure Data Factory (ADF)** and **Microsoft Fabric (Data Factory pipelines and Workflows)**.

---

## 1. The Core Purpose of Orchestration

While compute engines like Spark handle *how* data is processed, orchestrators handle *when*, *where*, and *under what conditions* it is processed. Orchestration provides:

* **Pipeline Scheduling & Triggers:** Running jobs on time-based schedules (e.g., nightly at 2:00 AM), tumbling windows, or event-based triggers (e.g., when a new file lands in ADLS Gen2).
* **Dependency Management:** Ensuring that Dimension tables finish loading successfully *before* Fact tables start, or running independent tasks in parallel.
* **Error Handling & Retries:** Automatically catching failures, sending alerts, or routing execution down failure paths if a script times out.
* **Parameterization & Metadata-Driven Pipelines:** Passing dynamic variables (like batch dates, watermark values, or source table names) into your code so a single pipeline can handle dozens of different datasets.

---

## 2. Azure Data Factory (ADF) vs. Microsoft Fabric

While ADF has been the classic enterprise orchestrator in Azure for years, Microsoft Fabric integrates these exact orchestration capabilities directly into a unified SaaS platform:

* **Azure Data Factory (ADF):** A standalone, cloud-native ETL/ELT service in Azure. It relies on linked services, integration runtimes (IRs), and decoupled storage/compute setups where you spin up Azure Databricks or HDInsight clusters to run your heavy Spark code.
* **Microsoft Fabric Pipelines:** Functionally identical to ADF pipelines in design and user experience, but natively integrated into the Fabric workspace. Instead of spinning up external compute linked services, Fabric pipelines can directly orchestrate native **Notebooks** (running Spark), Dataflows Gen2, and Copy Activities within the same SaaS environment.

---

## 3. Key Components of an Orchestration Pipeline

Whether you are in ADF or Fabric, pipelines are built using a visual or code-driven DAG (Directed Acyclic Graph) structure composed of specific activities:

* **Copy Activity:** The workhorse for data ingestion. It handles moving data from hundreds of connectors (SQL databases, APIs, Salesforce, S3) into your data lake with built-in partitioning and fault tolerance.
* **Notebook Activity:** Executes your PySpark or Scala notebooks. You can pass parameters from the pipeline (e.g., `@pipeline().parameters.ExecutionDate`) straight into the notebook cells using widget inputs.
* **Control Flow Activities:**
* **For Each / Until:** Loops through lists of tables or files to execute tasks iteratively.
* **If Condition / Switch:** Branches pipeline logic based on success, failure, or metadata values.
* **Web Activity:** Calls external REST APIs or triggers notifications (such as sending alerts to Microsoft Teams or Slack).



---

## 4. Metadata-Driven Architecture (The Enterprise Standard)

Instead of hardcoding a separate pipeline for every single table you want to ingest or transform, enterprise data engineers build **Metadata-Driven Pipelines**:

1. **Control Table:** A SQL database or configuration file stores a list of all tables, source connection strings, watermark columns, and target paths.
2. **Lookup Activity:** The pipeline reads this control table at runtime.
3. **ForEach Activity:** Iterates through the rows dynamically, passing the table name and parameters to a reusable copy or Spark notebook activity.

This approach allows you to onboard dozens of new data sources simply by adding a row to a database table, requiring zero changes to the pipeline structure.